# Project 11: Handwritten Digit Classification Using CNN

**Goal:** Build a basic Convolutional Neural Network (CNN) that recognises handwritten digits (0-9) from the MNIST dataset.

**Plan**
1. Load and preprocess the data
2. Build the CNN
3. Train it
4. Evaluate: accuracy, confusion matrix, classification report, sample predictions
5. Look at the mistakes
6. Conclusions

## Step 1: Load and preprocess the data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# fixed seeds so results are close to repeatable (not 100% identical on every machine)
tf.random.set_seed(42)
np.random.seed(42)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Train images:", x_train.shape, "| Train labels:", y_train.shape)
print("Test images: ", x_test.shape, "| Test labels: ", y_test.shape)
print("Pixel value range:", x_train.min(), "to", x_train.max())

Each image is 28x28 pixels in grayscale. Let's look at a few.

In [ ]:
plt.figure(figsize=(10, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

### Check class balance

If one digit had far fewer examples than the others, accuracy alone would be misleading.

In [ ]:
counts = np.bincount(y_train)
plt.figure(figsize=(7, 3))
plt.bar(range(10), counts)
plt.xticks(range(10))
plt.xlabel("Digit")
plt.ylabel("Number of training images")
plt.title("Training set class distribution")
plt.show()
print("Smallest class:", counts.min(), "| Largest class:", counts.max())

### Preprocessing

Two things to do:
1. **Scale pixels from 0-255 down to 0-1.** Neural networks train much better on small numbers.
2. **Add a channel dimension.** A CNN expects images shaped `(height, width, channels)`. MNIST is grayscale, so channels = 1.

I'm keeping the labels as plain integers (0-9) and using `sparse_categorical_crossentropy` as the loss, so no one-hot encoding is needed.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)   # (60000, 28, 28) -> (60000, 28, 28, 1)
x_test = np.expand_dims(x_test, -1)

print("New train shape:", x_train.shape)
print("Pixel value range now:", x_train.min(), "to", x_train.max())

### Validation split

I hold out the last 10% of the training data as a **validation set**. It is used to watch for overfitting while training. 
The 10,000 test images stay untouched until the final evaluation.

In [ ]:
val_size = 6000
x_val, y_val = x_train[-val_size:], y_train[-val_size:]
x_train_final, y_train_final = x_train[:-val_size], y_train[:-val_size]

print("Train:", x_train_final.shape[0], "| Validation:", x_val.shape[0], "| Test:", x_test.shape[0])

## Step 2: Build the CNN

Architecture (kept small and standard):

| Layer | Purpose |
|---|---|
| `Conv2D(32, 3x3)` + ReLU | Learns small local patterns like edges and strokes |
| `MaxPooling2D(2x2)` | Shrinks the image by half and keeps the strongest signals |
| `Conv2D(64, 3x3)` + ReLU | Combines edges into bigger shapes like curves and loops |
| `MaxPooling2D(2x2)` | Shrinks again |
| `Flatten` | Turns the feature maps into one long vector |
| `Dropout(0.5)` | Randomly switches off half the neurons during training to reduce overfitting |
| `Dense(10, softmax)` | Outputs a probability for each digit 0-9 |

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## Step 3: Train the model

I train for up to 10 epochs, with **early stopping** on validation loss so the model stops once it quits improving 
(and restores the best weights it saw).

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)

history = model.fit(
    x_train_final, y_train_final,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop],
    verbose=2,
)

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

Training accuracy can sit slightly *below* validation accuracy early on. That is normal here, because dropout is active during training but switched off when validating.

## Step 4: Evaluate on the test set

This is the first time the model sees the 10,000 test images.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss:     {test_loss:.4f}")

In [ ]:
# predicted probabilities -> predicted digit
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

### Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix (rows = true digit, columns = predicted digit)")
plt.show()

The diagonal is correct predictions. Any number off the diagonal is a mistake. 
The diagonal dominates the colour scale, so the errors are hard to see above. Here is the same matrix with the diagonal removed:

In [ ]:
cm_errors = cm.copy()
np.fill_diagonal(cm_errors, 0)

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm_errors, display_labels=range(10)).plot(ax=ax, cmap="Reds", colorbar=False)
ax.set_title("Misclassifications only")
plt.show()

In [ ]:
# top 5 most common confusions, as a table
pairs = [(cm_errors[i, j], i, j) for i in range(10) for j in range(10) if i != j]
pairs.sort(reverse=True)

print("Most common mistakes:")
for count, true_d, pred_d in pairs[:5]:
    print(f"  True {true_d} predicted as {pred_d}: {count} times")

### Classification report (precision, recall, F1 per digit)

In [ ]:
print(classification_report(y_test, y_pred, digits=3))

- **Precision:** of all images the model called "7", how many really were 7?
- **Recall:** of all real 7s, how many did the model find?
- **F1-score:** a single number combining the two.

### Per-digit comparison

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred)

x = np.arange(10)
w = 0.27
plt.figure(figsize=(10, 4))
plt.bar(x - w, prec, w, label="Precision")
plt.bar(x, rec, w, label="Recall")
plt.bar(x + w, f1, w, label="F1")
plt.xticks(x)
plt.ylim(min(prec.min(), rec.min()) - 0.02, 1.0)
plt.xlabel("Digit")
plt.title("Precision / Recall / F1 by digit (y-axis does not start at 0)")
plt.legend(loc="lower right")
plt.show()

worst = np.argmin(f1)
print(f"Lowest F1: digit {worst} ({f1[worst]:.3f})")

## Step 5: Sample predictions

In [ ]:
# 15 random test images with the model's prediction and confidence
rng = np.random.default_rng(0)
idx = rng.choice(len(x_test), 15, replace=False)

plt.figure(figsize=(12, 5))
for k, i in enumerate(idx):
    plt.subplot(3, 5, k + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    pred, conf = y_pred[i], y_pred_probs[i].max()
    colour = "green" if pred == y_test[i] else "red"
    plt.title(f"True {y_test[i]} | Pred {pred}\n({conf:.0%})", color=colour, fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

### Look at the actual mistakes

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print(f"Total misclassified: {len(wrong)} out of {len(y_test)}")

# show the 15 mistakes the model was most confident about
wrong_sorted = wrong[np.argsort(-y_pred_probs[wrong].max(axis=1))][:15]

plt.figure(figsize=(12, 5))
for k, i in enumerate(wrong_sorted):
    plt.subplot(3, 5, k + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.title(f"True {y_test[i]} | Pred {y_pred[i]}\n({y_pred_probs[i].max():.0%})", color="red", fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()

Many of these are hard even for a person: sloppy handwriting, unusual stroke styles, or digits that really do look like another one (a "6" that looks like an "l", a "5" that looks like a "3"). This is a useful check on whether the errors are "reasonable" or whether something is broken.

Also notice the confidence values. Some of these wrong predictions come with 99-100% confidence, so a high softmax probability does **not** mean the prediction is correct.

## Step 6: Conclusions

**Result:** A small CNN with two convolutional layers reaches roughly 98-99% accuracy on the 10,000 unseen test images after a few epochs.

**Confusion matrix:** The errors are not spread evenly. A handful of digit pairs account for most of the mistakes (see the "Most common mistakes" list above). Pairs with similar shapes tend to be the ones confused with each other.

**Classification report:** Precision, recall and F1 are all high for every digit, since the classes are balanced and the task is relatively easy. The digit with the lowest F1 is the one to look at first if you want to improve the model.

**Why a CNN works here:** Convolution layers look for local patterns (strokes, curves) no matter where they appear in the image, and pooling keeps the important signals while reducing size. This uses far fewer parameters than a fully connected network on raw pixels.

**Limitations**
- MNIST is very clean: centred, size-normalised digits. Real handwriting (off-centre, different pens, noisy backgrounds) would score lower.
- Results change slightly from run to run because of random weight initialisation and dropout.
- The model is often very confident when it is wrong, so its probabilities should not be treated as a reliability measure.
- Accuracy is one test run. I did not tune hyperparameters, so there is likely still some room to improve.

**Ideas to improve the model**
- Add a third convolution layer or more filters.
- Add data augmentation (small rotations, shifts) for robustness.
- Add batch normalisation.
- Train a few models with different seeds and average their predictions.